In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>เวอร์ชันแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirement ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
นอกจาก[การแสดงผล instruction บน circuit](/guides/visualize-circuits) แล้ว คุณอาจต้องการแสดงผลการจัดตาราง circuit โดยใช้เมธอด [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) ของ Qiskit การแสดงผลนี้ช่วยให้ระบุเวลาที่ Qubit ไม่ได้ทำงานได้อย่างรวดเร็ว อย่างไรก็ตาม เมธอดนี้ไม่คืนค่าที่ถูกต้องสำหรับ dynamic circuit ในการแสดงผลการจัดตาราง dynamic circuit ให้ใช้เมธอด `draw_circuit_schedule_timing` ตามที่อธิบายในส่วน [Qiskit Runtime support](#qr-support)

## ตัวอย่าง

ในการแสดงผลโปรแกรม circuit ที่จัดตารางแล้ว คุณสามารถเรียกใช้ฟังก์ชันนี้พร้อม control arguments ชุดหนึ่ง ลักษณะส่วนใหญ่ของรูปภาพผลลัพธ์สามารถปรับแต่งได้ผ่าน stylesheet แต่ไม่จำเป็นต้องทำ

### วาดด้วย stylesheet ค่าเริ่มต้น

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### วาดด้วย stylesheet สำหรับการดีบักโปรแกรม

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

คุณสามารถสร้างฟังก์ชัน generator หรือ layout แบบกำหนดเองและอัปเดต stylesheet ที่มีอยู่ด้วยฟังก์ชันเหล่านั้น วิธีนี้ช่วยให้คุณควบคุมลักษณะส่วนใหญ่ของรูปภาพผลลัพธ์ได้โดยไม่ต้องแก้ไข codebase ของ scheduled circuit drawer ดูตัวอย่างเพิ่มเติมได้ที่ API reference ของ [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer)
<span id="qr-support"></span>
## Qiskit Runtime support
แม้ว่า timeline drawer ที่ built-in ใน Qiskit จะมีประโยชน์สำหรับ static circuit แต่อาจไม่ได้แสดงผล timing ของ [dynamic circuit](/guides/classical-feedforward-and-control-flow) อย่างถูกต้อง เนื่องจาก implicit operation เช่น broadcasting และการกำหนด branch ในฐานะส่วนหนึ่งของการรองรับ dynamic circuit Qiskit Runtime จะคืนข้อมูล timing ของ circuit ที่ถูกต้องภายใน job result เมื่อมีการร้องขอ

> **Note:** - นี่คือฟังก์ชันทดลอง อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง
> - ฟังก์ชันนี้ใช้ได้เฉพาะกับ Qiskit Runtime Sampler job เท่านั้น
> - แม้ว่า circuit time รวมจะถูกคืนใน metadata ของ "compilation" แต่นี่ไม่ใช่เวลาที่ใช้สำหรับการเรียกเก็บเงิน (quantum time)
### เปิดใช้งานการดึงข้อมูล timing
ในการเปิดใช้งานการดึงข้อมูล timing ให้ตั้งค่า flag ทดลอง `scheduler_timing` เป็น `True` เมื่อรัน primitive job